# Introduction
The dataset consists of numerical samples used for supervised learning.
Each row represents one observation, identified by a unique ID.

## Structure

### Training Set:
$$ID, INPUT_0, INPUT_1, ..., INPUT_{N-1}, TARGET_0, TARGET_1, TARGET_2, TARGET_3$$
Contains both input features and corresponding target values.

So $$X = (x_i) = (INPUT_0, INPUT_1, ..., INPUT_{N-1}) \quad i=1..500, N=12$$ and $$y_i=(TARGET_0, TARGET_1, TARGET_2, TARGET_3)\quad i=1..500$$

### Blind Test Set:
$$ID, INPUT_1, INPUT_2, ..., INPUT_N$$
Contains only input features; target values are omitted.

# 1. Training set analysis

In [ ]:
## in the ML-CUP25-TR.csv comment is explained the shape of the dataset
n_inputs = 12

## Training set: ID, INPUTS, TARGET_1, TARGET_2, TARGET_3, TARGET_4 (last 4 columns)
columns = (
    ["ID"] +
    [f"INPUT_{i}" for i in range(n_inputs)] +
    [f"TARGET_{i}" for i in range(4)]
)

In [ ]:
import pandas as pd
import numpy as np

ml_cup_tr = pd.read_csv("./data/MLC25/ML-CUP25-TR.csv", skiprows=7, names=columns)

In [ ]:
ml_cup_tr.shape

In [ ]:
ml_cup_tr.describe()

In [ ]:
X_tr = ml_cup_tr[[f"INPUT_{i}" for i in range(n_inputs)]].values
y_tr = ml_cup_tr[[f"TARGET_{i}" for i in range(4)]].values

In [ ]:
X_tr.shape

In [ ]:
y_tr.shape

## Preprocessing

The golden rule is
> - If your model is sensitive to feature magnitude (like neural networks, KNN, gradient boosting): 
>   - MinMaxScaler is usually best. 
> - If your model assumes normal distribution or uses distance metrics, 
>   - StandardScaler is usually better.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import *

In [ ]:
def plot_dataset_scaling(X, label):
    n_features = X.shape[1]
    for i in range(n_features):
        plt.hist(X[:, i], alpha=0.4, label=f"input_{i}")
    plt.legend(loc='upper right', ncol=3)
    if label:
        plt.title(label)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_dataset_scaling(X_tr, "Original TR")

### 1. MinMaxScaler
- Rescales features to a fixed range (usually 0–1 or -1–1). 
  - Preserves the shape of the original distribution but compresses values.
- Use when all features should have equal weight and you know the min/max bounds (e.g., neural networks).

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_tr)
X_tr_scaled = scaler.transform(X_tr)

In [ ]:
plot_dataset_scaling(X_tr_scaled, "MinMax scaling TR")

In [ ]:
pd.DataFrame(X_tr).describe()

In [ ]:
pd.DataFrame(X_tr_scaled).describe()

### 2. StandardScaler

- Centers data at mean 0 with unit variance (Z-score normalization). 
  - Keeps the relative shape, but shifts and scales by mean and standard deviation.
- Use when data may contain outliers or unknown range, or algorithms assume normality (e.g., linear/logistic regression, PCA, SVM).

In [ ]:
scaler = StandardScaler()
scaler.fit(X_tr)
X_tr_scaled = scaler.transform(X_tr)

In [ ]:
plot_dataset_scaling(X_tr_scaled, "Standard scaling TR")

In [ ]:
pd.DataFrame(X_tr).describe()

In [ ]:
# apply(...) to suppress scientific notation, otherwise is confusonary
pd.DataFrame(X_tr_scaled).describe().apply(lambda s: s.apply(lambda x: format(x, 'g')))

Capiamo se ci sono tanti outlier utilizzando il Z-score, visto che ora abbiamo dati normalizzati.

In [ ]:
# |Z| = |(x - mu) / sigma|
abs_z_scores = np.abs(X_tr_scaled)

# 2. Crea una maschera booleana per identificare i punti che superano la soglia
# True dove |Z| > soglia
outlier_mask = abs_z_scores > 3

# 3. Conta il numero totale di True nella maschera
# np.sum() funziona anche per array 2D/multi-dimensionali, restituendo il totale complessivo.
numero_totale_outlier = np.sum(outlier_mask)

# 4. Trova gli indici dei punti outlier
# np.where() restituisce gli indici (righe, colonne) dove la condizione è True.
indici_outlier = np.where(outlier_mask)

In [ ]:
numero_totale_outlier, indici_outlier

Possiamo concludere che non ci sono outlier

# 2. Test set analysis

In [ ]:
## in the ML-CUP25-TS.csv comment is explained the shape of the dataset
n_inputs = 12

## Test set: ID, INPUTS
columns = (
    ["ID"] +
    [f"INPUT_{i}" for i in range(n_inputs)]
)

In [ ]:
import pandas as pd

ml_cup_ts = pd.read_csv("./data/MLC25/ML-CUP25-TS.csv", skiprows=7, names=columns)

In [ ]:
ml_cup_ts.describe()

In [ ]:
X_ts = ml_cup_ts[[f"INPUT_{i}" for i in range(n_inputs)]].values

In [ ]:
X_ts.shape

## Preprocessing

The golden rule is
> - If your model is sensitive to feature magnitude (like neural networks, KNN, gradient boosting): 
>   - MinMaxScaler is usually best. 
> - If your model assumes normal distribution or uses distance metrics, 
>   - StandardScaler is usually better.

In [ ]:
plot_dataset_scaling(X_ts, "Original TS")

## 1. MinMax Scaling

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_ts)
X_ts_scaled = scaler.transform(X_ts)

In [ ]:
plot_dataset_scaling(X_tr_scaled, "MinMax scaling TS")

In [ ]:
pd.DataFrame(X_ts).describe()

In [ ]:
pd.DataFrame(X_ts_scaled).describe()

### 2. StandardScaler

In [ ]:
scaler = StandardScaler()
scaler.fit(X_ts)
X_ts_scaled = scaler.transform(X_ts)

In [ ]:
plot_dataset_scaling(X_tr_scaled, "Standard scaling TS")

In [ ]:
pd.DataFrame(X_ts).describe()

In [ ]:
# apply(...) to suppress scientific notation, otherwise is confusonary
pd.DataFrame(X_tr_scaled).describe().apply(lambda s: s.apply(lambda x: format(x, 'g')))